In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

print('pandas', pd.__version__)


pandas 3.0.5


In [2]:
RAW = Path('../raw/philly_rentals_raw.csv')

if not RAW.exists():
    RAW = Path('philly_rentals_raw.csv')

raw = pd.read_csv(RAW)
df = raw.copy()

print(RAW.resolve())
print(df.shape)


/Users/k1nghandy/git/learning/wcupa/CSC381-01/CSC381/hw/hw-1/raw/philly_rentals_raw.csv
(31, 6)


In [3]:
def rows_changed(before, after, what):
    lost = before - after
    pct = 100 * lost / before if before else 0
    print(f'{what}: {before} -> {after} rows ({lost} removed, {pct:.1f}%)')
    return after


In [4]:
df.head(10)


,listing_id,neighborhood,bedrooms,bathrooms,sqft,rent
0,R-2001,Fishtown,1,1.0,620,"$1,450"
1,R-2002,Manayunk,2,1.0,890,1750
2,R-2003,University City,1,1.0,540,"$1,395"
3,R-2004,fishtown,2,1.0,845,"1,900"
4,R-2005,Northern Liberties,3,2.0,1240,"$2,750"
5,R-2006,Old City,1,1.0,600,1650
6,R-2007,Manayunk,NaN,1.0,710,"$1,525"
7,R-2008,FISHTOWN,2,1.5,930,"2,050"
8,R-2009,University City,1,1.0,820,"$1,875"
9,R-2010,Old City,2,2.0,980,2400


In [5]:
df.tail(3)


,listing_id,neighborhood,bedrooms,bathrooms,sqft,rent
28,R-2029,University City,2,1.5,835,"$1,990"
29,R-2030,Old City,3,2.0,1265,"2,850"
30,TOTAL,NaN,NaN,NaN,26385,"$54,395"


In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   listing_id    31 non-null     str    
 1   neighborhood  30 non-null     str    
 2   bedrooms      28 non-null     str    
 3   bathrooms     30 non-null     float64
 4   sqft          31 non-null     int64  
 5   rent          31 non-null     str    
dtypes: float64(1), int64(1), str(4)
memory usage: 1.6 KB


In [7]:
df['neighborhood'].value_counts()


neighborhood
Manayunk              6
University City       6
Old City              6
Northern Liberties    5
Fishtown              3
fishtown              2
FISHTOWN              1
Fishtown              1
Name: count, dtype: int64

In [8]:
df['bedrooms'].value_counts()


bedrooms
2          11
1          10
3           6
unknown     1
Name: count, dtype: int64

In [9]:
df['bathrooms'].value_counts()


bathrooms
1.0    20
2.0     7
1.5     3
Name: count, dtype: int64

In [10]:
df['sqft'].value_counts()


sqft
620      1
890      1
540      1
845      1
1240     1
600      1
710      1
930      1
820      1
980      1
575      1
1310     1
610      1
650      1
560      1
870      1
665      1
905      1
1150     1
590      1
915      1
860      1
1195     1
525      1
940      1
605      1
1080     1
880      1
835      1
1265     1
26385    1
Name: count, dtype: int64

In [11]:
df['rent'].value_counts()


rent
$1,450     1
1750       1
$1,395     1
1,900      1
$2,750     1
1650       1
$1,525     1
2,050      1
$1,875     1
2400       1
$1,600     1
2,395      1
$1,475     1
1550       1
$1,700     1
1,950      1
$1,400     1
1825       1
$2,300     1
1,625      1
$2,100     1
1700       1
$2,650     1
1,340      1
$0         1
1,575      1
$2,250     1
1,880      1
$1,990     1
2,850      1
$54,395    1
Name: count, dtype: int64

*Part 1 — what is wrong with this file*

1. 6 — 'bedrooms' R-2007 is missing bedroom data.
2. 24 – 'rent' value_counts – One listing is missing rent amount
3. 'rent' value_counts – Rent amounts include comma divider and $ for every other entry.
4. 3, 7 – Neighborhood case changes – some lowercase, most Uppercase, some all capital.


In [ ]:
# 1. Drop row that is not a listing
print("Before:", len(df), "rows in")

df = df[df["listing_id"] != "TOTAL"].copy()

print("After:", len(df), "rows out")


Before: 30 rows in
After: 30 rows out


In [15]:
# 2. Make `rent` a real number.
df["rent"] = (
    df["rent"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

print(df["rent"].dtype)


float64


In [16]:
# 3. Clean `neighborhood`.
df["neighborhood"] = (
    df["neighborhood"]
    .str.strip()
    .str.title()
)

print(df["neighborhood"].nunique())
print(df["neighborhood"].value_counts())


5
neighborhood
Fishtown              7
Manayunk              6
University City       6
Old City              6
Northern Liberties    5
Name: count, dtype: int64


In [ ]:
# Part 3 – `df['bedrooms'].isna().sum()`
df["bedrooms"] = df["bedrooms"].replace(
    ["N/A", "unknown", ""],
    np.nan
)

df["bedrooms"] = pd.to_numeric(df["bedrooms"])

print(df["bedrooms"].isna().sum())


3


In [19]:
df["bedrooms"] = df["bedrooms"].fillna(
    df.groupby("neighborhood")["bedrooms"].transform("median")
)

print(df["bedrooms"].isna().sum()) # 0


0


In [20]:
df.to_csv("rentals_clean.csv", index=False)


In [21]:
import sqlite3
con = sqlite3.connect(':memory:')
df.to_sql('listings', con, index=False)

def q(sql):
    return pd.read_sql(sql, con)
